In [ ]:
!pip install librosa

In [ ]:
import numpy as np
import pandas as pd
import librosa
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import layers, models

In [ ]:
DATA_PATH = "/content/audio_dataset"

labels = []
features = []

def extract_features(file):
    audio, sr = librosa.load(file, duration=3)
    mfcc = np.mean(librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40).T, axis=0)
    return mfcc

for file in os.listdir(DATA_PATH):
    if file.endswith(".wav"):
        path = os.path.join(DATA_PATH, file)
        feature = extract_features(path)
        if "fear" in file or "angry" in file:
            labels.append("alert")
        else:
            labels.append("normal")
        features.append(feature)

X = np.array(features)
y = np.array(labels)

In [ ]:
le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
tiny_model = models.Sequential([
    layers.Dense(32, activation='relu', input_shape=(40,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

tiny_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history_tiny = tiny_model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

In [ ]:
plt.plot(history_tiny.history['accuracy'], label='train acc')
plt.plot(history_tiny.history['val_accuracy'], label='val acc')
plt.legend()
plt.title("TinyML Model Accuracy")
plt.show()

In [ ]:
X_cnn = X.reshape(X.shape[0], 40, 1)

X_train, X_test, y_train, y_test = train_test_split(X_cnn, y, test_size=0.2)

cnn_model = models.Sequential([
    layers.Conv1D(64, 3, activation='relu', input_shape=(40,1)),
    layers.MaxPooling1D(2),
    layers.Conv1D(128, 3, activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history_cnn = cnn_model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

In [ ]:
plt.plot(history_cnn.history['accuracy'], label='train acc')
plt.plot(history_cnn.history['val_accuracy'], label='val acc')
plt.legend()
plt.title("CNN Model Accuracy")
plt.show()